# Constrained Viterbi with MVRs

This notebook demonstrates `viterbi_torch_mvr_chmm`, the constrained Viterbi
decoder for hidden Markov models with **mediation variable representation (MVR)**
constraints.

An MVR is a state machine over hidden-state sequences. It carries an
auxiliary *mediation* state alongside the HMM's hidden state:

| component | meaning |
| --- | --- |
| `ini: h -> m` | mediation state entered when the automaton starts on hidden state `h` |
| `upd: (m_prev, h_curr) -> m_curr` | transition |
| `evl: m -> bool` | acceptance predicate |

A hidden sequence is feasible iff the terminal MVR evaluates under `evl` as `True`. Viterbi then searches the **augmented** state space
(hidden state x mediation state) and returns the most likely *augmented* path. The mediation path is a deterministic function of the hidden path, and the augmented posterior is degenerate in the mediation part. Therefore, the hidden part of the most likely augmented path is exactly the most likely constrained path.

An MVR may also carry a `time_range = [start, end]`, the segment over which the
constraint is enforced. Within that window the automaton is initialized at
`start` and evaluated at `end`; outside it, the MVR contributes nothing at all.

We look at three cases:

1. a global "never visit state `A`" constraint,
2. the same constraint restricted to a `time_range`,
3. a mix — one global constraint plus one windowed constraint.

Each is compared against unconstrained Viterbi (`conin.hidden_markov_model.inference.viterbi`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

from conin.hidden_markov_model.hmm import HiddenMarkovModel
from conin.hidden_markov_model.inference import viterbi
from conin.hidden_markov_model.mvr import HomMVR
from conin.hidden_markov_model.chmm_mvr import MVR_CHMM
from conin.hidden_markov_model.inference.viterbi_mvr import viterbi_torch_mvr_chmm

## 1. The model

A three-state HMM over hidden states `A`, `B`, `C` emitting `lo` / `mid` / `hi`.
The parameters are written out explicitly so the notebook is fully reproducible.

The structure worth noting: `C` emits `hi` almost always (p = 0.90), `A` is the
most likely predecessor and successor of `C`, and `B` is comparatively unlikely
to be entered. That makes `A` and `C` the natural path through a `hi`-heavy
observation sequence, so forbidding `A` will force a visible detour.

In [ ]:
HIDDEN_STATES = ["A", "B", "C"]
OBSERVED_STATES = ["lo", "mid", "hi"]

start_probs = {
    "A": 0.2765440507007986,
    "B": 0.4033576072467887,
    "C": 0.32009834205241255,
}

transition_probs = {
    ("A", "A"): 0.3391777054270445,
    ("A", "B"): 0.049711711669595204,
    ("A", "C"): 0.6111105829033604,
    ("B", "A"): 0.48102507253852517,
    ("B", "B"): 0.05601918704283972,
    ("B", "C"): 0.4629557404186351,
    ("C", "A"): 0.43616112524444134,
    ("C", "B"): 0.1773076392327265,
    ("C", "C"): 0.38653123552283214,
}

emission_probs = {
    ("A", "lo"): 0.19949219710155375,
    ("A", "mid"): 0.30789837305397333,
    ("A", "hi"): 0.492609429844473,
    ("B", "lo"): 0.534907622618408,
    ("B", "mid"): 0.234417585356662,
    ("B", "hi"): 0.23067479202493,
    ("C", "lo"): 0.09093879934300991,
    ("C", "mid"): 0.008996844382398088,
    ("C", "hi"): 0.9000643562745919,
}

hmm = HiddenMarkovModel()
hmm.load_model(
    start_probs=start_probs,
    transition_probs=transition_probs,
    emission_probs=emission_probs,
    initialize=True,
)

print("hidden states  :", hmm.hidden_states)
print("observed states:", hmm.observed_states)

display(
    pd.DataFrame(
        [[transition_probs[(a, b)] for b in HIDDEN_STATES] for a in HIDDEN_STATES],
        index=pd.Index(HIDDEN_STATES, name="from"),
        columns=pd.Index(HIDDEN_STATES, name="to"),
    ).style.format("{:.3f}").set_caption("Transition probabilities")
)

display(
    pd.DataFrame(
        [[emission_probs[(h, o)] for o in OBSERVED_STATES] for h in HIDDEN_STATES],
        index=pd.Index(HIDDEN_STATES, name="hidden"),
        columns=pd.Index(OBSERVED_STATES, name="emits"),
    ).style.format("{:.3f}").set_caption("Emission probabilities")
)

## 2. The observation sequence

Ten observations, mostly `hi`, ending in two `lo`s.

In [ ]:
observed = ["mid", "hi", "hi", "hi", "hi", "mid", "hi", "hi", "lo", "lo"]
T = len(observed)

print("t   :", "  ".join(f"{t:>3}" for t in range(T)))
print("obs :", "  ".join(f"{o:>3}" for o in observed))

## 3. Baseline: unconstrained Viterbi

`conin.hidden_markov_model.inference.viterbi` is conin's existing unconstrained
decoder. It returns a `Munch` whose `solution.hidden` holds the most likely path
in external labels and whose `solution.log_likelihood` holds its log probability.

In [ ]:
unconstrained = viterbi(observed=observed, hmm=hmm)

path_unconstrained = unconstrained.solution.hidden
ll_unconstrained = unconstrained.solution.log_likelihood

print("path          :", " ".join(path_unconstrained))
print("log-likelihood:", f"{ll_unconstrained:.4f}")

# Sanity check that the two code paths agree on how a path is scored.
print("hmm.log_probability:", f"{hmm.log_probability(observed, path_unconstrained):.4f}")

The unconstrained path visits `A` at **t = 0, 5, 9** and `C` everywhere else. Both
of the states we are about to restrict are in active use, so every constraint
below will change the answer.

## 4. Expressing "never visit `A`" as an MVR

The automaton needs two mediation states. It starts in `violated` if the very
first hidden state is the forbidden one, and `violated` is absorbing:

```
ini(h)            = violated  if h == forbidden else ok
upd(m_prev, h)    = violated  if m_prev == violated or h == forbidden else ok
evl(ok)           = True
evl(violated)     = False
```

`time_range` is passed straight through to `HomMVR`. Leaving it `None` enforces
the constraint over the whole sequence.

In [ ]:
def forbid_state_mvr(forbidden_state, time_range=None):
    """Build an MVR rejecting any path that visits `forbidden_state` in its window."""
    mediation_states = ["ok", "violated"]

    return HomMVR(
        hidden_states=HIDDEN_STATES,
        mediation_states=mediation_states,
        ini={
            h: ("violated" if h == forbidden_state else "ok") for h in HIDDEN_STATES
        },
        upd={
            (m, h): ("violated" if m == "violated" or h == forbidden_state else "ok")
            for m in mediation_states
            for h in HIDDEN_STATES
        },
        evl={"ok": True, "violated": False},
        time_range=time_range,
    )


demo_mvr = forbid_state_mvr("A")

print("mediation states:", demo_mvr.mediation_states)
print("ini             :", demo_mvr.ini)
print("evl             :", demo_mvr.evl)
print("time_range      :", demo_mvr._time_range)

## 5. A helper for running constrained Viterbi

The MVRs are attached to the HMM through `MVR_CHMM`, then decoded by
`viterbi_torch_mvr_chmm`. With `return_augmented=True` the decoder also reports,
for every time step, which mediation state each active MVR was in.

In [ ]:
def solve(constraints, label):
    """Decode the most likely feasible path and report it against the baseline."""
    model = MVR_CHMM(hidden_markov_model=hmm, constraints=constraints)

    path, augmented, score = viterbi_torch_mvr_chmm(
        model, observed, return_augmented=True, return_score=True
    )

    print(f"{label}")
    print(f"  unconstrained : {' '.join(path_unconstrained)}   ll = {ll_unconstrained:.4f}")
    print(f"  constrained   : {' '.join(path)}   ll = {score:.4f}")
    print(f"  cost of the constraint: {ll_unconstrained - score:.4f} nats")

    return {"label": label, "path": path, "augmented": augmented, "score": score}

# Constrained Inference Examples

## Example 1 — global "don't visit `A`"

No `time_range`, so the constraint is enforced over the entire sequence.

In [ ]:
example_1 = solve([forbid_state_mvr("A")], "Example 1: never visit A")

assert "A" not in example_1["path"]

`A` is gone everywhere. At t = 0 and t = 5 the decoder falls back to `B`, a poor
emitter for these observations; at t = 9 it prefers `C` instead.

## Example 2 — "don't visit `A`" restricted to a time range

The same constraint, now with `time_range = [3, 6]`. The MVR is initialized at
t = 3, run forward, and evaluated at t = 6. Before t = 3 and after t = 6 it is
**absent from the lattice entirely** — it holds no mediation state and places no
factor on the path.

In [ ]:
WINDOW = [3, 6]

example_2 = solve(
    [forbid_state_mvr("A", time_range=WINDOW)],
    f"Example 2: never visit A on t in {WINDOW}",
)

inside = example_2["path"][WINDOW[0] : WINDOW[1] + 1]
outside = [h for t, h in enumerate(example_2["path"]) if not WINDOW[0] <= t <= WINDOW[1]]

print()
print(f"  inside  {WINDOW}: {' '.join(inside)}   -> contains A? {'A' in inside}")
print(f"  outside {WINDOW}: {' '.join(outside)}   -> contains A? {'A' in outside}")

assert "A" not in inside
assert "A" in outside

This is the behaviour that distinguishes a windowed MVR: `A` is banned at t = 5,
which falls inside the window, but is freely used at t = 0 and t = 9, which do
not. Because the constraint binds on a shorter stretch, it costs only about
1.5 nats instead of 2.4.

## Example 3 — one global constraint plus one subsequence constraint

Two MVRs at once: `A` is banned everywhere, and `C` is additionally banned on
`[3, 6]`. The two automata run side by side on their own axes of the augmented
lattice; no product automaton is constructed.

In [ ]:
example_3 = solve(
    [forbid_state_mvr("A"), forbid_state_mvr("C", time_range=WINDOW)],
    f"Example 3: never visit A, and never visit C on t in {WINDOW}",
)

inside_3 = example_3["path"][WINDOW[0] : WINDOW[1] + 1]

print()
print(f"  A anywhere?          {'A' in example_3['path']}")
print(f"  C inside {WINDOW}?    {'C' in inside_3}")
print(f"  C outside the window? {'C' in [h for t, h in enumerate(example_3['path']) if not WINDOW[0] <= t <= WINDOW[1]]}")

assert "A" not in example_3["path"]
assert "C" not in inside_3

With `A` unavailable everywhere and `C` unavailable inside the window, the only
option left on `[3, 6]` is `B` — a state the model strongly prefers to avoid on
`hi` observations. The likelihood falls by roughly 12 nats, far more than either
constraint costs alone.

## 6. Visualizing the paths

Red bands mark the forbidden (state, time) region for each scenario. The grey
dashed line is the unconstrained path; the blue line is the constrained one.

In [ ]:
STATE_Y = {h: i for i, h in enumerate(HIDDEN_STATES)}

SCENARIOS = [
    (example_1, [("A", (0, T - 1))]),
    (example_2, [("A", tuple(WINDOW))]),
    (example_3, [("A", (0, T - 1)), ("C", tuple(WINDOW))]),
]


def draw_scenario(ax, result, forbidden_regions):
    for state, (a, b) in forbidden_regions:
        ax.add_patch(
            Rectangle(
                (a - 0.5, STATE_Y[state] - 0.42),
                (b - a) + 1,
                0.84,
                facecolor="crimson",
                alpha=0.16,
                edgecolor="crimson",
                linewidth=0.8,
                zorder=0,
            )
        )

    ax.plot(
        range(T),
        [STATE_Y[h] for h in path_unconstrained],
        "--o",
        color="0.6",
        markersize=5,
        linewidth=1.2,
        label="unconstrained",
        zorder=2,
    )
    ax.plot(
        range(T),
        [STATE_Y[h] for h in result["path"]],
        "-o",
        color="#2E6DB4",
        markersize=7,
        linewidth=2.0,
        label="constrained",
        zorder=3,
    )

    # Circle the time steps where the two disagree.
    differing = [t for t in range(T) if result["path"][t] != path_unconstrained[t]]
    ax.plot(
        differing,
        [STATE_Y[result["path"][t]] for t in differing],
        "o",
        markerfacecolor="none",
        markeredgecolor="#C44E00",
        markersize=14,
        markeredgewidth=2,
        zorder=4,
    )

    ax.set_yticks(range(len(HIDDEN_STATES)))
    ax.set_yticklabels(HIDDEN_STATES)
    ax.set_ylim(-0.6, len(HIDDEN_STATES) - 0.4)
    ax.set_xlim(-0.6, T - 0.4)
    ax.set_xticks(range(T))
    ax.grid(axis="x", color="0.9", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.set_ylabel("hidden state")
    ax.set_title(
        f"{result['label']}    (log-likelihood {result['score']:.3f})",
        loc="left",
        fontsize=11,
    )


fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)

for ax, (result, regions) in zip(axes, SCENARIOS):
    draw_scenario(ax, result, regions)

axes[-1].set_xlabel("time")
axes[-1].set_xticklabels([f"{t}\n{o}" for t, o in enumerate(observed)])

# A figure-level legend keeps it clear of the data in every panel.
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=2, frameon=False)

fig.suptitle(
    "Constrained vs unconstrained Viterbi  "
    "(red = forbidden state-time region, orange ring = differs from baseline)",
    fontsize=12,
)
fig.tight_layout(rect=(0, 0.04, 1, 1))
plt.show()

Reading the middle panel against the top one shows the effect of `time_range`
directly: the red band shrinks to `[3, 6]`, and the blue path immediately
reclaims `A` at t = 0 and t = 9 where the constraint no longer applies.

### All four paths together

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

series = [("unconstrained", path_unconstrained, "0.5", ll_unconstrained)] + [
    (r["label"].split(":")[0], r["path"], c, r["score"])
    for r, c in zip(
        [example_1, example_2, example_3], ["#2E6DB4", "#2CA25F", "#C44E00"]
    )
]

for i, (name, path, color, score) in enumerate(series):
    offset = (i - 1.5) * 0.10
    ax.plot(
        range(T),
        [STATE_Y[h] + offset for h in path],
        "-o",
        color=color,
        markersize=6,
        linewidth=1.8,
        label=f"{name}  ({score:.2f})",
        alpha=0.9,
    )

ax.axvspan(WINDOW[0] - 0.5, WINDOW[1] + 0.5, color="0.92", zorder=0)
ax.text(
    (WINDOW[0] + WINDOW[1]) / 2,
    len(HIDDEN_STATES) - 0.45,
    f"time_range {WINDOW}",
    ha="center",
    fontsize=9,
    color="0.35",
)

ax.set_yticks(range(len(HIDDEN_STATES)))
ax.set_yticklabels(HIDDEN_STATES)
ax.set_ylim(-0.6, len(HIDDEN_STATES) - 0.3)
ax.set_xticks(range(T))
ax.set_xticklabels([f"{t}\n{o}" for t, o in enumerate(observed)])
ax.set_xlabel("time")
ax.set_ylabel("hidden state")
ax.grid(axis="x", color="0.93", linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), title="path (log-likelihood)")
ax.set_title("Decoded paths under each constraint set", loc="left")

fig.tight_layout()
plt.show()

## 7. Summary

In [ ]:
summary = pd.DataFrame(
    [
        {
            "scenario": name,
            "path": " ".join(path),
            "log-likelihood": score,
            "cost vs unconstrained": ll_unconstrained - score,
        }
        for name, path, score in [
            ("unconstrained", path_unconstrained, ll_unconstrained),
            ("1: global !A", example_1["path"], example_1["score"]),
            (f"2: !A on {WINDOW}", example_2["path"], example_2["score"]),
            (f"3: global !A + !C on {WINDOW}", example_3["path"], example_3["score"]),
        ]
    ]
)

display(
    summary.style.format(
        {"log-likelihood": "{:.4f}", "cost vs unconstrained": "{:.4f}"}
    ).hide(axis="index")
)

## 8. Inspecting the mediation states

`return_augmented=True` exposes the automaton's internal state at every step.
This makes the `time_range` semantics visible: outside `time_range` the MVR has no
entry at all, because augmentation happens only over the given range.

Below, MVR `0` is the windowed "never visit `A`" constraint from Example 2.

In [ ]:
rows = []

for t, entry in enumerate(example_2["augmented"]):
    rows.append(
        {
            "t": t,
            "observed": observed[t],
            "hidden": example_2["path"][t],
            "mvr 0 mediation state": entry["mvr_states"].get(0, "-- not enforced --"),
        }
    )

display(pd.DataFrame(rows).style.hide(axis="index"))

The automaton appears only on `t = 3..6`. It is initialized at t = 3 from the
hidden state there, runs forward, and is evaluated at t = 6 — where it must be
`ok`. Steps 0-2 and 7-9 carry no mediation state, which is exactly why `A`
remains available at t = 0 and t = 9.

For Example 3 both constraints appear, each on its own axis and each over its own
window:

In [ ]:
rows = []

for t, entry in enumerate(example_3["augmented"]):
    rows.append(
        {
            "t": t,
            "observed": observed[t],
            "hidden": example_3["path"][t],
            "mvr 0 (global !A)": entry["mvr_states"].get(0, "--"),
            f"mvr 1 (!C on {WINDOW})": entry["mvr_states"].get(1, "--"),
        }
    )

display(pd.DataFrame(rows).style.hide(axis="index"))

## 9. A user-defined horizon and intermittent observations

Both the horizon and the observations are decoupled from each other.
`observed` may be a dense list *or* a sparse `{time: label}` map, and
`time_horizon` sets how many steps to decode over — possibly more than there
are observations.

A time with no observation is **not** dropped from the chain. It still consumes
a transition and still drives every MVR active there; only its emission factor
goes away (it becomes `1`, or `0` in log space). So the decoder still commits to
a hidden state at that time, chosen by the transitions and the constraints
alone.

In [ ]:
# Throw away all but three of the ten observations.
KEPT = [0, 4, 9]
sparse_observed = {t: observed[t] for t in KEPT}

model_free = MVR_CHMM(hidden_markov_model=hmm, constraints=[])

path_dense, ll_dense = viterbi_torch_mvr_chmm(
    model_free, observed, return_augmented=False, return_score=True
)
path_sparse, ll_sparse = viterbi_torch_mvr_chmm(
    model_free, sparse_observed, time_horizon=T,
    return_augmented=False, return_score=True,
)

print("t             :", "  ".join(f"{t:>3}" for t in range(T)))
print("observation   :", "  ".join(f"{o:>3}" for o in observed))
print("kept          :", "  ".join(f"{('^' if t in KEPT else ' '):>3}" for t in range(T)))
print()
print("all 10 obs    :", "  ".join(f"{h:>3}" for h in path_dense), f"  ll = {ll_dense:.4f}")
print("only 3 obs    :", "  ".join(f"{h:>3}" for h in path_sparse), f"  ll = {ll_sparse:.4f}")
print()
print(f"both paths have length {len(path_sparse)} -- dropping an observation does not "
      "drop the time step")

The sparse path still has one state per time step — that is the point. But note
that it disagrees with the dense decode almost everywhere, *including at the
three times that are still observed*.

Nothing pins an observed time to the state it took before. Removing seven
emission factors changes the objective, and what is left is dominated by the
transition matrix, which strongly favours alternating `A` and `C`
(`A→C = 0.611`, `C→A = 0.436`). The decoder happily gives up a good emission
match at `t = 0` and `t = 9` to buy a better-scoring chain of transitions.

The two log-likelihoods are not comparable to each other, incidentally: they
score different quantities, since the sparse run's product simply contains fewer
emission factors.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.4))

ax.plot(range(T), [STATE_Y[h] for h in path_dense], "--o", color="0.6",
        markersize=5, linewidth=1.2, label="all 10 observations", zorder=2)
ax.plot(range(T), [STATE_Y[h] for h in path_sparse], "-o", color="#2E6DB4",
        markersize=7, linewidth=2.0, label="only t = 0, 4, 9 observed", zorder=3)

for t in KEPT:
    ax.axvline(t, color="#2E6DB4", alpha=0.18, linewidth=8, zorder=0)

differing = [t for t in range(T) if path_dense[t] != path_sparse[t]]
ax.plot(differing, [STATE_Y[path_sparse[t]] for t in differing], "o",
        markerfacecolor="none", markeredgecolor="#C44E00", markersize=14,
        markeredgewidth=2, zorder=4)

ax.set_yticks(range(len(HIDDEN_STATES)))
ax.set_yticklabels(HIDDEN_STATES)
ax.set_ylim(-0.6, len(HIDDEN_STATES) - 0.4)
ax.set_xlim(-0.6, T - 0.4)
ax.set_xticks(range(T))
ax.set_xticklabels([f"{t}\n{o}" for t, o in enumerate(observed)])
ax.grid(axis="x", color="0.9", linewidth=0.8)
ax.set_axisbelow(True)
ax.set_ylabel("hidden state")
ax.set_xlabel("time")
ax.set_title("Intermittent observations  (blue bands = observed times, "
             "orange ring = differs from the fully observed decode)",
             loc="left", fontsize=11)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.32), ncol=2, frameon=False)

fig.tight_layout()
plt.show()

In [ ]:
# Decode four steps past the end of the observations, under the global "never A".
EXTENDED = T + 4

model_noA = MVR_CHMM(hidden_markov_model=hmm, constraints=[forbid_state_mvr("A")])

path_ext, ll_ext = viterbi_torch_mvr_chmm(
    model_noA, observed, time_horizon=EXTENDED,
    return_augmented=False, return_score=True,
)

print("t        :", "  ".join(f"{t:>3}" for t in range(EXTENDED)))
print("obs      :", "  ".join(f"{(observed[t] if t < T else '-'):>3}" for t in range(EXTENDED)))
print("path     :", "  ".join(f"{h:>3}" for h in path_ext))
print()
print(f"log-likelihood      : {ll_ext:.4f}")
print(f"'A' anywhere in path: {'A' in path_ext}")
print()
print("The unobserved tail is still constrained: an MVR with no time_range")
print(f"defaults to [0, time_horizon - 1] = [0, {EXTENDED - 1}], not to the observed span.")

That defaulting is worth remembering, because it is the one visible behaviour
change when `time_horizon` is passed explicitly:

- an MVR with no `time_range` is enforced over the **extended** horizon, and
- an `InhomMVR` that was long enough for `len(observed)` may now be rejected as
  too short.

Both follow from `time_range` defaulting to `[0, T-1]` against the resolved
horizon rather than the observation count. Give the MVR an explicit
`time_range` when you want it pinned to the observed span.

## Notes

- `viterbi_torch_mvr_chmm` runs in the max-plus semiring (log probabilities), so
  the reported score is directly comparable to `viterbi`'s `log_likelihood` and
  to `hmm.log_probability`.
- Constraints are kept as separate axes of the augmented state space rather than
  being multiplied into a single product automaton. Adding a second MVR in
  Example 3 added an axis; it did not rebuild the first one.
- If a constraint set admits no feasible path at all, the decoder raises
  `InvalidInputError` rather than returning a violating path.
- The MVRs used here are homogeneous (`HomMVR`). Time-inhomogeneous constraints
  (`InhomMVR`), whose mediation space and transitions vary with time, are
  also supported.
- `observed` accepts a dense list or a sparse `{time: label}` map, and
  `time_horizon` decouples the number of decoded steps from the number of
  observations. Unobserved times keep their place in the chain and are scored by
  the transitions alone.